# Advanced 10 — Sufficiency, Completeness, and UMVU Estimation

Practice notebook for the **"Sufficiency, Completeness, and UMVU Estimation"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Sufficiency and minimal sufficiency

A statistic \( T(X) \) is **sufficient** for \( \theta \) if the conditional
distribution of \( X \) given \( T(X) \) does not depend on \( \theta \). A
sufficient statistic is **minimal sufficient** if it is a function of every other
sufficient statistic.

The **factorization theorem** gives a practical criterion: \( T \) is sufficient
iff the joint density factorizes as

$$
f(x\mid\theta) = h(x)\, g(T(x), \theta).
$$

**Running example — exponential family.** Let
\( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Exp}(\lambda) \) with rate
\( \lambda \), density \( f(x\mid\lambda)=\lambda e^{-\lambda x} \). Then

$$
f(x\mid\lambda) = \lambda^n \exp\!\left(-\lambda \sum_{i=1}^n X_i\right)
= \underbrace{1}_{h(x)}\,
\underbrace{\lambda^n e^{-\lambda T(x)}}_{g(T,\lambda)}, \qquad
T(x)=\sum_i X_i.
$$

So \( T=\sum_i X_i \) is sufficient for \( \lambda \). It is in fact minimal
sufficient. A direct consequence: the conditional distribution of \( X_1/T \) given
\( T=t \) must not depend on \( \lambda \). In fact
\( X_1/T \mid T=t \sim \mathrm{Beta}(1, n-1) \) for all \( \lambda \). We verify
this numerically by drawing samples at two very different rates and overlaying the
conditional histogram of \( X_1/T \) given \( T \) close to a fixed value \( t_0 \).


In [ ]:
def conditional_ratio_sim(lam, n, t0, tol, M=2_000_000, seed=0):
    """Draw Exp(lam) samples, keep those with T in [t0-tol, t0+tol], return X1/T."""
    rng = np.random.default_rng(seed)
    X = rng.exponential(1.0 / lam, size=(M, n))
    T = X.sum(axis=1)
    mask = np.abs(T - t0) < tol
    return X[mask, 0] / T[mask]

n = 5
t0 = 3.0
tol = 0.05

# Same conditional distribution despite very different rates -> T is sufficient
for lam in [0.5, 2.0, 8.0]:
    r = conditional_ratio_sim(lam, n, t0, tol, seed=int(lam * 100))
    plt.hist(r, bins=60, density=True, alpha=0.45, label=f"lam={lam}, |T-t0|<{tol}")

# Beta(1, n-1) reference density
grid = np.linspace(0, 1, 400)
plt.plot(grid, stats.beta.pdf(grid, 1, n - 1), "k-", lw=2, label="Beta(1, n-1)")
plt.xlabel(r"$X_1/T \mid T \approx t_0$"); plt.ylabel("density")
plt.title(f"Conditional law does not depend on lambda (n={n}, t0={t0})")
plt.legend(); plt.show()

print("All three histograms collapse onto the same Beta(1, n-1) curve: ")
print("the conditional distribution is free of lambda, so T = sum X_i is sufficient.")


---
## Part 2 — Rao–Blackwell theorem

Let \( T \) be sufficient for \( \theta \) and \( \delta(X) \) any estimator
of \( \phi(\theta) \) with finite risk under squared loss. The **Rao–Blackwell
improvement** is

$$
\delta^*(X) = E_\theta[\,\delta(X) \mid T\,].
$$

Then \( \delta^* \) has risk no larger than \( \delta \) (and strictly smaller
unless \( \delta \) is already a function of \( T \)). Intuitively, conditioning
on \( T \) averages out the noise that carries no information about \( \theta \).

**Concrete demo.** Let \( X_1,\dots,X_n \stackrel{iid}{\sim} N(\theta, 1) \).
The crude unbiased estimator \( \delta(X)=X_1 \) has
\( \mathrm{Var}=1 \). The sample mean \( T=\bar X \) is sufficient for
\( \theta \). Rao–Blackwellizing:

$$
\delta^*(X) = E[X_1 \mid \bar X] = \bar X,
$$

since by exchangeability \( E[X_1\mid \bar X]=\bar X \). Its variance is
\( 1/n \) — an \( n \)-fold reduction. We verify by Monte Carlo: estimate
\( \mathrm{Var}(X_1) \) and \( \mathrm{Var}(\bar X) \) and confirm
\( \mathrm{Var}(\delta^*) \leq \mathrm{Var}(\delta) \).


In [ ]:
def rb_variance_demo(theta, n, M=500_000, seed=0):
    """Monte Carlo variance of the crude estimator X1 and its Rao-Blackwellization Xbar."""
    rng = np.random.default_rng(seed)
    X = rng.normal(theta, 1.0, size=(M, n))
    crude = X[:, 0]            # delta  = X_1
    rb     = X.mean(axis=1)    # delta* = E[X_1 | Xbar] = Xbar
    return crude.var(ddof=0), rb.var(ddof=0)

theta = 2.0
for n in [2, 5, 10, 20, 50]:
    v_crude, v_rb = rb_variance_demo(theta, n)
    print(f"n={n:3d}  Var(X1)={v_crude:.5f} (theory 1.00000)  "
          f"Var(Xbar)={v_rb:.5f} (theory {1/n:.5f})  "
          f"ratio={v_crude/v_rb:.2f}")

# Distribution picture: X_1 is wide, Xbar is tight
rng = np.random.default_rng(123)
M = 200_000
n = 10
X = rng.normal(theta, 1.0, size=(M, n))
crude = X[:, 0]
rb    = X.mean(axis=1)
plt.hist(crude, bins=80, density=True, alpha=0.5, label=r"$\delta=X_1$")
plt.hist(rb,    bins=80, density=True, alpha=0.5, label=r"$\delta^*=\bar X$")
plt.axvline(theta, color="k", ls="--", label=r"true $\theta$")
plt.xlabel("estimate"); plt.ylabel("density")
plt.title(f"Rao-Blackwell: Var {1:.2f} -> {1/n:.2f} (n={n})")
plt.legend(); plt.show()

print("Rao-Blackwell cannot increase variance: conditioning on a sufficient")
print("statistic discards noise while preserving the unbiased mean.")


---
## Part 3 — Lehmann–Scheffé: UMVU estimator of \( e^{-\lambda} \) for Poisson

A statistic \( T \) is **complete** for \( \{P_\theta\} \) if

$$
E_\theta[g(T)] = 0 \;\forall\theta \quad\Rightarrow\quad
P_\theta(g(T)=0)=1 \;\forall\theta.
$$

**Lehmann–Scheffé.** If \( T \) is complete sufficient for \( \theta \) and
\( \hat\phi(T) \) is unbiased for \( \phi(\theta) \), then \( \hat\phi(T) \)
is the **unique UMVU** estimator of \( \phi(\theta) \).

**Target.** Let \( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Pois}(\lambda) \).
We want to estimate \( \phi(\lambda)=e^{-\lambda}=P(X_i=0) \) (e.g. the probability
of zero events in one period).

- \( S=\sum_i X_i \sim \mathrm{Pois}(n\lambda) \) is complete sufficient for
  \( \lambda \) (full-rank exponential family).
- The crude unbiased estimator is \( \delta(X)=\mathbf 1\{X_1=0\} \), since
  \( E[\delta]=P(X_1=0)=e^{-\lambda} \). Its variance is
  \( e^{-\lambda}(1-e^{-\lambda}) \).
- Rao–Blackwellize on \( S \). Given \( S=s \), the count \( X_1 \) is
  \( \mathrm{Binomial}(s, 1/n) \), so
  \( P(X_1=0\mid S=s)=(1-1/n)^s \). Therefore
  \( \delta^*(X)=E[\delta\mid S]=(1-1/n)^S \).

Because \( S \) is complete and sufficient and \( \delta^* \) is unbiased,
Lehmann–Scheffé says \( (1-1/n)^S \) is **UMVU** for \( e^{-\lambda} \). Its
variance is, using the Poisson pgf
\( E[t^S]=\exp(n\lambda(t-1)) \),

$$
\mathrm{Var}\!\left((1-1/n)^S\right)
= E[(1-1/n)^{2S}] - e^{-2\lambda}
= e^{n\lambda((1-1/n)^2-1)} - e^{-2\lambda}
= e^{-2\lambda}\!\left(e^{\lambda/n}-1\right).
$$

We compare \( \mathrm{Var}(\delta^*) \) to \( \mathrm{Var}(\delta) \) both
analytically and by simulation.


In [ ]:
def umvu_poisson_demo(lam, n, M=2_000_000, seed=0):
    """Compare crude 1{X1=0} vs UMVU (1-1/n)^S for estimating e^{-lam}."""
    rng = np.random.default_rng(seed)
    X = rng.poisson(lam, size=(M, n))
    crude = (X[:, 0] == 0).astype(float)
    S = X.sum(axis=1)
    umvu = (1.0 - 1.0 / n) ** S
    target = np.exp(-lam)
    return (crude.mean(), umvu.mean(), target,
            crude.var(ddof=0), umvu.var(ddof=0))

n = 10
print(f"{'lam':>6} {'target':>10} {'E[crude]':>10} {'E[umvu]':>10} "
      f"{'Var(crude)':>12} {'Var(umvu)':>12} {'theory crude':>14} {'theory umvu':>14}")
for lam in [0.2, 0.5, 1.0, 2.0, 4.0]:
    mc, um, tgt, vc, vu = umvu_poisson_demo(lam, n, seed=int(lam*100))
    th_crude = np.exp(-lam) * (1 - np.exp(-lam))
    th_umvu  = np.exp(-2*lam) * (np.expm1(lam/n))   # exp(-2lam)*(exp(lam/n)-1)
    print(f"{lam:>6.2f} {tgt:>10.5f} {mc:>10.5f} {um:>10.5f} "
          f"{vc:>12.6f} {vu:>12.6f} {th_crude:>14.6f} {th_umvu:>14.6f}")

# Plot variance ratio as a function of lambda
lams = np.linspace(0.05, 5, 120)
v_crude_th = np.exp(-lams) * (1 - np.exp(-lams))
v_umvu_th  = np.exp(-2*lams) * np.expm1(lams / n)
plt.plot(lams, v_crude_th, label=r"Var$\,\mathbf 1\{X_1=0\}$ (crude)")
plt.plot(lams, v_umvu_th,  label=r"Var$\,(1-1/n)^S$ (UMVU)")
plt.xlabel(r"$\lambda$"); plt.ylabel("estimator variance")
plt.title(rf"UMVU vs crude for $e^{{-\lambda}}$ (n={n})")
plt.legend(); plt.show()

print("UMVU variance is uniformly below the crude variance: Lehmann-Scheffé")
print("delivers the unique minimum-variance unbiased estimator via a complete sufficient statistic.")


---
## Part 4 — Completeness, numerically

Completeness is the property that powers Lehmann–Scheffé. For
\( S\sim \mathrm{Pois}(n\lambda) \),

$$
E_\lambda[g(S)] = \sum_{s=0}^\infty g(s)\, e^{-n\lambda}\frac{(n\lambda)^s}{s!}
= e^{-n\lambda}\sum_{s=0}^\infty \frac{g(s)}{s!}(n\lambda)^s.
$$

If \( E_\lambda[g(S)]=0 \) for every \( \lambda>0 \), the power series in
\( \lambda \) is identically zero, so every coefficient \( g(s)/s! \) is zero,
i.e. \( g\equiv 0 \). Hence \( S \) is complete.

We verify this numerically by discretizing the problem: truncate \( s=0,\dots,K \)
and pick \( K+1 \) distinct values \( \lambda_j \). The matrix
\( A_{j,s}=P_{\lambda_j}(S=s) \) should have **full column rank** — its only null
vector is \( g=0 \). Equivalently, the smallest singular value of \( A \) is
bounded away from zero. We build \( A \) and inspect its singular values and null
space, confirming that no nontrivial unbiased estimator of zero sits on \( S \).


In [ ]:
def completeness_matrix(lams, K, n):
    """A[j, s] = P_{lam_j}(S = s) for S ~ Pois(n * lam_j), s = 0..K."""
    A = np.zeros((len(lams), K + 1))
    for j, lam in enumerate(lams):
        mu = n * lam
        s = np.arange(K + 1)
        A[j, :] = stats.poisson.pmf(s, mu)
    return A

n = 10
K = 8
lams = np.linspace(0.1, 4.0, K + 1)
A = completeness_matrix(lams, K, n)

# Smallest singular value: large => columns independent => only g=0 solves A g = 0
sv = np.linalg.svd(A, compute_uv=False)
print("singular values of A:")
for i, s in enumerate(sv):
    print(f"  sigma_{i} = {s:.4e}")
print(f"smallest singular value = {sv[-1]:.4e}  (>> 0 means trivial null space)")

# Attempt to find a nontrivial g with E_lam[g(S)] = 0 at all lam_j: solve A g = 0.
# Use lstsq on the homogeneous system A g ~ 0 with ||g|| = 1 by SVD; the minimizer
# is the right-singular vector of the smallest singular value.
_, _, Vt = np.linalg.svd(A)
g_min = Vt[-1]                      # candidate null vector
resid = A @ g_min
print(f"\ncandidate g (smallest right-singular vector):")
print("  ", np.array2string(g_min, precision=4))
print(f"||A g||_2 = {np.linalg.norm(resid):.4e}  (should be ~ sigma_min, NOT ~0)")

# Scale so E[g(S)] ~ 0 at each lam_j has unit-norm residuals; compare to ||A||.
print(f"||A||_2  = {sv[0]:.4e}")
print(f"ratio ||A g|| / ||A|| = {np.linalg.norm(resid)/sv[0]:.4e}  (not zero => no nontrivial null)")

# As a sanity check, the trivial g = 0 does satisfy A g = 0 exactly:
print(f"\n||A * 0|| = {np.linalg.norm(A @ np.zeros(K+1)):.4e}  (only the zero vector works)")

# Grow K: the smallest singular value stays bounded away from 0 as we add points,
# demonstrating completeness is not an artifact of truncation.
print("\nsmallest singular value vs K:")
for Kk in [4, 6, 8, 10, 12]:
    lk = np.linspace(0.1, 4.0, Kk + 1)
    Ak = completeness_matrix(lk, Kk, n)
    sk = np.linalg.svd(Ak, compute_uv=False)[-1]
    print(f"  K={Kk:2d}  sigma_min = {sk:.4e}")


---
## Your turn

- For \( X_i\sim \mathrm{Exp}(\lambda) \), show that \( T=\sum_i X_i \) is
  minimal sufficient by arguing that the ratio
  \( f(x\mid\lambda)/f(y\mid\lambda) \) is free of \( \lambda \) iff
  \( \sum_i x_i = \sum_i y_i \). Reproduce the conditional-histogram check of
  Part 1 with a different \( t_0 \) and \( n=8 \).
- Rao–Blackwellize the estimator \( \delta(X)=(X_1+X_2)/2 \) for \( \theta \)
  in the \( N(\theta,1) \) model. What do you get? Verify the variance reduction
  by Monte Carlo.
- Find the UMVU estimator of \( \phi(\lambda)=\lambda e^{-\lambda} \)
  (the probability of exactly one Poisson count) from a sample
  \( X_1,\dots,X_n\sim \mathrm{Pois}(\lambda) \) using the complete sufficient
  statistic \( S=\sum X_i \). Check unbiasedness and compare variance to the
  crude estimator \( \mathbf 1\{X_1=1\} \).
- Build the completeness matrix \( A \) for \( \bar X \) in the
  \( N(\mu,1) \) model truncated to a fine grid of \( \mu \)-values and confirm
  numerically that its smallest singular value stays positive as the grid is
  refined.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Use the factorization theorem to show that for \( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Exp}(\lambda) \) the sum \( T=\sum_i X_i \) is sufficient for \( \lambda \). Then show \( T \) is minimal sufficient by proving the ratio \( f(x\mid\lambda)/f(y\mid\lambda) \) is independent of \( \lambda \) iff \( T(x)=T(y) \).

2. Let \( X_1,\dots,X_n \stackrel{iid}{\sim} N(\theta,1) \). Consider the crude unbiased estimator \( \delta(X)=X_1 \) of \( \theta \). State the Rao–Blackwell improvement \( \delta^*(X)=E[X_1\mid \bar X] \) and derive its variance. Verify \( \mathrm{Var}(\delta^*) \leq \mathrm{Var}(\delta) \) by Monte Carlo for \( n=5,10,25 \).

3. For \( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Pois}(\lambda) \), the complete sufficient statistic is \( S=\sum_i X_i \). Show that \( \hat\phi(S)=(1-1/n)^S \) is unbiased for \( e^{-\lambda} \) using the Poisson probability generating function \( E[t^S]=\exp(n\lambda(t-1)) \), and derive its variance. Confirm by simulation that it beats the crude estimator \( \mathbf 1\{X_1=0\} \).

4. Define a complete statistic and state the Lehmann–Scheffé theorem. Then verify numerically that \( S\sim \mathrm{Pois}(n\lambda) \) is complete by building the matrix \( A_{j,s}=P_{\lambda_j}(S=s) \) on a grid of \( \lambda_j \) values and showing its smallest singular value is bounded away from zero as the truncation level \( K \) grows.

5. Find the UMVU estimator of \( \phi(\lambda)=\lambda e^{-\lambda} \) for \( X_1,\dots,X_n\sim \mathrm{Pois}(\lambda) \). (Hint: Rao–Blackwellize \( \mathbf 1\{X_1=1\} \) on \( S \); given \( S=s \), \( X_1\mid S=s\sim \mathrm{Bin}(s,1/n) \).) Verify unbiasedness by Monte Carlo and compare its variance to the crude estimator.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The joint density is
\( f(x\mid\lambda)=\lambda^n e^{-\lambda\sum_i x_i} \), which factorizes as
\( h(x)=1 \) times \( g(T,\lambda)=\lambda^n e^{-\lambda T(x)} \) with
\( T(x)=\sum_i x_i \), so \( T \) is sufficient. For minimal sufficiency,
\( f(x\mid\lambda)/f(y\mid\lambda)=\exp[-\lambda(T(x)-T(y))] \), which is free
of \( \lambda \) iff \( T(x)=T(y) \). Hence \( T \) is minimal sufficient.

```python
import numpy as np
from scipy import stats
n = 8; lam = 2.0; t0 = 4.0; tol = 0.05
rng = np.random.default_rng(0)
X = rng.exponential(1/lam, size=(2_000_000, n))
T = X.sum(axis=1); m = np.abs(T - t0) < tol
r = X[m, 0] / T[m]
grid = np.linspace(0, 1, 400)
plt.hist(r, bins=60, density=True, alpha=0.5, label="sim")
plt.plot(grid, stats.beta.pdf(grid, 1, n-1), "k-", lw=2, label="Beta(1, n-1)")
plt.legend(); plt.show()
```

**2.** By exchangeability \( E[X_1\mid\bar X]=\bar X \). So
\( \delta^*(X)=\bar X \) with \( \mathrm{Var}=1/n \), versus
\( \mathrm{Var}(X_1)=1 \). Rao–Blackwell gives a factor-\( n \) variance
reduction.

```python
import numpy as np
theta = 1.5; M = 500_000
for n in [5, 10, 25]:
    rng = np.random.default_rng(n)
    X = rng.normal(theta, 1.0, size=(M, n))
    v_crude = X[:, 0].var()
    v_rb    = X.mean(axis=1).var()
    print(f"n={n:2d}  Var(X1)={v_crude:.4f}  Var(Xbar)={v_rb:.4f}  "
          f"theory 1/n={1/n:.4f}  reduction={v_crude/v_rb:.1f}x")
```
You should observe \( \mathrm{Var}(\bar X)\approx 1/n \) and
\( \mathrm{Var}(X_1)/\mathrm{Var}(\bar X)\approx n \).

**3.** Using the pgf \( E[t^S]=\exp(n\lambda(t-1)) \),
\( E[(1-1/n)^S]=\exp(n\lambda((1-1/n)-1))=\exp(-\lambda)=e^{-\lambda} \),
so it is unbiased. Also
\( E[(1-1/n)^{2S}]=\exp(n\lambda((1-1/n)^2-1))=\exp(-2\lambda+\lambda/n) \).
Thus
\( \mathrm{Var}=e^{-2\lambda}(e^{\lambda/n}-1) \), versus
\( \mathrm{Var}(\mathbf 1\{X_1=0\})=e^{-\lambda}(1-e^{-\lambda}) \).

```python
import numpy as np
n = 10; lam = 1.0; M = 2_000_000
rng = np.random.default_rng(0)
X = rng.poisson(lam, size=(M, n))
S = X.sum(axis=1)
crude = (X[:, 0] == 0).astype(float)
umvu = (1 - 1/n) ** S
print(f"E[crude]={crude.mean():.5f}  E[umvu]={umvu.mean():.5f}  target={np.exp(-lam):.5f}")
print(f"Var(crude)={crude.var():.6f}  Var(umvu)={umvu.var():.6f}  "
      f"theory {np.exp(-lam)*(1-np.exp(-lam)):.6f} / {np.exp(-2*lam)*np.expm1(lam/n):.6f}")
```
The UMVU variance is strictly smaller — Lehmann–Scheffé in action.

**4.** Completeness: \( E_\lambda[g(S)]=0\,\forall\lambda\Rightarrow g\equiv 0 \).
With \( A_{j,s}=P_{\lambda_j}(S=s) \), \( Ag=0 \) encodes
\( E_{\lambda_j}[g(S)]=0 \) on a grid; a full-rank \( A \) forces \( g=0 \).

```python
import numpy as np
n = 10
def completeness_matrix(lams, K, n):
    A = np.zeros((len(lams), K + 1))
    for j, lam in enumerate(lams):
        mu = n * lam; s = np.arange(K + 1)
        A[j, :] = stats.poisson.pmf(s, mu)
    return A
for K in [4, 6, 8, 10, 12]:
    lams = np.linspace(0.1, 4.0, K + 1)
    A = completeness_matrix(lams, K, n)
    smin = np.linalg.svd(A, compute_uv=False)[-1]
    print(f"K={K:2d}  sigma_min={smin:.4e}")
```
\( \sigma_{\min} \) stays well above zero as \( K \) grows, so the only null
vector is \( g=0 \): \( S \) is complete.

**5.** Given \( S=s \), \( X_1\mid S=s\sim \mathrm{Bin}(s,1/n) \), so
\( P(X_1=1\mid S=s)=\binom{s}{1}(1/n)(1-1/n)^{s-1}=\frac{s}{n}(1-1/n)^{s-1} \) for
\( s\geq 1 \), and \( 0 \) for \( s=0 \). Hence
\( \hat\phi(S)=\frac{S}{n}(1-1/n)^{S-1} \) is unbiased for
\( \lambda e^{-\lambda} \) and, by Lehmann–Scheffé, UMVU.

```python
import numpy as np
n = 10; lam = 1.0; M = 2_000_000
rng = np.random.default_rng(0)
X = rng.poisson(lam, size=(M, n))
S = X.sum(axis=1)
umvu = np.where(S > 0, (S/n) * (1 - 1/n)**(S - 1), 0.0)
crude = (X[:, 0] == 1).astype(float)
target = lam * np.exp(-lam)
print(f"target={target:.5f}  E[crude]={crude.mean():.5f}  E[umvu]={umvu.mean():.5f}")
print(f"Var(crude)={crude.var():.6f}  Var(umvu)={umvu.var():.6f}")
```
The UMVU estimator has lower variance than the indicator, as guaranteed by
Lehmann–Scheffé.


</details>
